# Sales Forecasting Notebook

This notebook demonstrates a simple demand forecasting workflow using historical sales data. It includes: 
- data cleaning and time-based feature engineering
- model training with a time-series-aware regression approach
- model evaluation and error analysis
- business-friendly forecast visualization and interpretation

## 1. Setup and data preparation

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

sales = pd.DataFrame([
    { 'date': '2022-01-01', 'revenue': 310 },
    { 'date': '2022-02-01', 'revenue': 285 },
    { 'date': '2022-03-01', 'revenue': 340 },
    { 'date': '2022-04-01', 'revenue': 390 },
    { 'date': '2022-05-01', 'revenue': np.nan },
    { 'date': '2022-06-01', 'revenue': 420 },
    { 'date': '2022-07-01', 'revenue': 460 },
    { 'date': '2022-08-01', 'revenue': 470 },
    { 'date': '2022-09-01', 'revenue': 520 },
    { 'date': '2022-10-01', 'revenue': 590 },
    { 'date': '2022-11-01', 'revenue': 650 },
    { 'date': '2022-12-01', 'revenue': 780 },
    { 'date': '2023-01-01', 'revenue': 320 },
    { 'date': '2023-02-01', 'revenue': 300 },
    { 'date': '2023-03-01', 'revenue': 380 },
    { 'date': '2023-04-01', 'revenue': 430 },
    { 'date': '2023-05-01', 'revenue': 510 },
    { 'date': '2023-06-01', 'revenue': 540 },
    { 'date': '2023-07-01', 'revenue': 590 },
    { 'date': '2023-08-01', 'revenue': 620 },
    { 'date': '2023-09-01', 'revenue': 670 },
    { 'date': '2023-10-01', 'revenue': 740 },
    { 'date': '2023-11-01', 'revenue': 820 },
    { 'date': '2023-12-01', 'revenue': 950 },
    { 'date': '2024-01-01', 'revenue': 340 },
    { 'date': '2024-02-01', 'revenue': 320 },
    { 'date': '2024-03-01', 'revenue': 395 },
    { 'date': '2024-04-01', 'revenue': 445 },
    { 'date': '2024-05-01', 'revenue': 530 },
    { 'date': '2024-06-01', 'revenue': 560 },
])
sales['date'] = pd.to_datetime(sales['date'])
sales = sales.set_index('date').asfreq('MS')
sales['revenue'] = sales['revenue'].interpolate(method='linear')
sales['month'] = sales.index.month
sales['year'] = sales.index.year
sales['lag_12'] = sales['revenue'].shift(12)
sales['trend'] = np.arange(len(sales))
sales['month_sin'] = np.sin(2 * np.pi * sales['month'] / 12)
sales['month_cos'] = np.cos(2 * np.pi * sales['month'] / 12)
sales = sales.dropna().copy()
sales.head()

## 2. Build the forecasting model

In [ ]:
features = ['lag_12', 'trend', 'month_sin', 'month_cos']
X = sales[features]
y = sales['revenue']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.18, shuffle=False)
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
r2 = model.score(X_test, y_test)
print(f'MAPE: {mape:.2f}%')
print(f'MAE: {mae:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'R^2: {r2:.3f}')

## 3. Forecast the next 12 months

In [ ]:
last_date = sales.index[-1] + pd.DateOffset(months=1)
future = []
history = sales.copy()
for _ in range(12):
    next_month = last_date + pd.DateOffset(months=len(future))
    lag_12 = history.loc[next_month - pd.DateOffset(months=12), 'revenue']
    trend = len(history) + len(future)
    month = next_month.month
    row = {
        'lag_12': lag_12,
        'trend': trend,
        'month_sin': np.sin(2 * np.pi * month / 12),
        'month_cos': np.cos(2 * np.pi * month / 12)
    }
    forecast_value = model.predict(pd.DataFrame([row]))[0]
    future.append({
        'date': next_month,
        'forecast': forecast_value,
        'lower': forecast_value - 45,
        'upper': forecast_value + 45
    })
future_df = pd.DataFrame(future).set_index('date')
future_df

## 4. Visualize actuals and forecast

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(sales.index, sales['revenue'], label='Actual Sales', marker='o')
plt.plot(future_df.index, future_df['forecast'], label='Forecast', marker='o', linestyle='--')
plt.fill_between(future_df.index, future_df['lower'], future_df['upper'], alpha=0.2, label='Confidence band')
plt.title('Sales Forecast for the Next 12 Months')
plt.xlabel('Month')
plt.ylabel('Revenue')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Business interpretation

- The forecast predicts how monthly revenue will evolve for the next year based on historical seasonality and trend.
- If the forecast shows higher summer or holiday months, a store owner can prepare inventory and staffing accordingly.
- If the model predicts a slowdown, a founder can adjust marketing spend or promotions to avoid cash-flow gaps.
- The dashboard provides clear KPIs such as MAPE and RMSE so decision-makers understand forecast confidence.